In [1]:
from enum import Enum
from pydantic import BaseModel, Field
from langchain_community.chat_models import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, get_buffer_string
from langchain_core.prompts import MessagesPlaceholder

## Step 1

In [2]:
class RequestType(str, Enum):
    QUESTION = "question"
    TASK = "task"
    SMALL_TALK = "small_talk"
    COMPLAINT = "complaint"
    UNKNOWN = "unknown"

class Classification(BaseModel):
    request_type: RequestType = Field(description="One of the predefined request types")
    confidence: float = Field(ge=0, le=1, description="Confidence score from 0 to 1")
    reasoning: str = Field(description="Brief justification for the chosen category")

class AssistantResponse(BaseModel):
    content: str = Field(description="The actual text of the assistant's reply")
    request_type: RequestType = Field(description="The request type used for this response")
    confidence: float = Field(description="Confidence from the classification stage")
    tokens_used: int = Field(description="Approximate count of tokens used")

### Test

In [3]:
try:
    valid_data = Classification(
        request_type=RequestType.QUESTION,
        confidence=0.95,
        reasoning="User asked for a definition"
    )
    print("Test 1 Passed:", valid_data)

    invalid_data = Classification(
        request_type="question",
        confidence=1.5,
        reasoning="error"
    )
except Exception as e:
    print("\nTest caught an error")
    print(e)

Test 1 Passed: request_type=<RequestType.QUESTION: 'question'> confidence=0.95 reasoning='User asked for a definition'

Test caught an error
1 validation error for Classification
confidence
  Input should be less than or equal to 1 [type=less_than_equal, input_value=1.5, input_type=float]
    For further information visit https://errors.pydantic.dev/2.12/v/less_than_equal


## Stpe 2

In [4]:
model = ChatOllama(model="llama3.1", temperature=0)

parser = PydanticOutputParser(pydantic_object=Classification)

classifier_prompt = ChatPromptTemplate.from_messages([
    ("system", """Ты - высокоточный классификатор намерений. Твоя задача - проанализировать запрос пользователя и определить его тип.
    
    ОГРАНИЧЕНИЕ: Ты должен писать обоснование (поле reasoning) на русском языке, если прямым текстом не указано иное.
    Исключением является генерируемый тобой код, который должен исключать использования русского языка
    
    Типы запросов:
    - question: поиск фактической информации или объяснение концепций (пример: "Что такое декоратор?")
    - task: конкретная просьба выполнить действие или сгенерировать контент (пример: "Напиши код")
    - small_talk: приветствия, прощания и светская беседа (пример: "Как дела?")
    - complaint: выражение недовольства, критика или описание проблемы (пример: "Ты тормозишь")
    - unknown: если запрос не поддается классификации или является бессмыслицей (пример: "kupila mama konika a konik bez nogi")

    Примеры (Few-shot):
    Запрос: "Привет!" -> small_talk
    Запрос: "Что такое LCEL?" -> question
    Запрос: "Расскажи анекдот" -> task
    Запрос: "Это плохой ответ" -> complaint
    Запрос: "kupila mama konika a konik bez nogi" -> unknown

    {format_instructions}
    
    ОТВЕЧАЙ ТОЛЬКО В ФОРМАТЕ JSON. ПИШИ ОБОСНОВАНИЕ (REASONING) НА РУССКОМ."""),
    ("human", "Запрос: {query}")
])

classifier_chain = (
    {"query": RunnablePassthrough(), "format_instructions": lambda _: parser.get_format_instructions()}
    | classifier_prompt
    | model
    | parser
)

def get_classification(user_input: str) -> Classification:
    try:
        return classifier_chain.invoke(user_input)
    except Exception as e:
        return Classification(
            request_type=RequestType.UNKNOWN,
            confidence=0.5,
            reasoning=f"Ошибка парсинга ответа модели: {str(e)}"
        )

/tmp/ipykernel_4513/1679400811.py:1: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import ChatOllama``.
  model = ChatOllama(model="llama3.1", temperature=0)


### Test

In [5]:
test_cases = [
    "Йоу!",
    "Объясни, как работает декоратор в Python",
    "Напиши функцию для вычисления чисел Фибоначчи",
    "Мне не нравится, как ты отвечаешь",
    "хыхыхыыхы"
]

for text in test_cases:
    print(f"--- Тест: {text} ---")
    result = get_classification(text)
    print(f"Тип: {result.request_type}")
    print(f"Уверенность: {result.confidence}")
    print(f"Обоснование: {result.reasoning}\n")

--- Тест: Йоу! ---
Тип: RequestType.SMALL_TALK
Уверенность: 1.0
Обоснование: Запрос содержит приветствие "Йоу!", что является типичным для светской беседы

--- Тест: Объясни, как работает декоратор в Python ---
Тип: RequestType.QUESTION
Уверенность: 1.0
Обоснование: Запрос содержит вопрос о работе декоратора в Python, что указывает на то, что пользователь ищет объяснение концепции. Декоратор - это функция, которая принимает другую функцию как аргумент и возвращает новую функцию с некоторыми модифицированными свойствами. В этом случае пользователь, вероятно, хочет понять принцип работы декоратора и его применение в программировании.

--- Тест: Напиши функцию для вычисления чисел Фибоначчи ---
Тип: RequestType.TASK
Уверенность: 1.0
Обоснование: Запрошено выполнение конкретной задачи - написание функции для вычисления чисел Фибоначчи. Вопрос не требует объяснения концепций или поиска фактической информации, а также не является выражением недовольства или приветствием.

--- Тест: Мне не нр

## Step 3


In [6]:
handler_prompts = {
    RequestType.QUESTION: "Ты - экспертный помощник. Дай информативный и полезный ответ на вопрос. Если не знаешь ответа - честно скажи об этом.",
    RequestType.TASK: "Ты - исполнительный ассистент. Пользователь просит выполнить задачу. Сделай это качественно, следуя всем инструкциям.",
    RequestType.SMALL_TALK: "Ты - дружелюбный собеседник. Поддерживай беседу, будь приветлив. Если пользователь представился - запомни его имя.",
    RequestType.COMPLAINT: "Ты - эмпатичный менеджер поддержки. Прояви сочувствие, постарайся понять суть проблемы и предложи конструктивное решение.",
    RequestType.UNKNOWN: "Ты - вежливый ассистент. Запрос пользователя неясен. Пожалуйста, вежливо попроси уточнить, что именно имел в виду пользователь."
}

def create_handler(request_type: RequestType):
    prompt_text = handler_prompts.get(request_type, handler_prompts[RequestType.UNKNOWN])
    prompt = ChatPromptTemplate.from_messages([
        ("system", prompt_text),
        ("human", "{query}")
    ])
    return prompt | model | StrOutputParser()

handlers = {rt: create_handler(rt) for rt in RequestType}

In [7]:
def route_and_handle(user_input: str):
    classification = get_classification(user_input)
    selected_handler = handlers.get(classification.request_type)
    response_text = selected_handler.invoke({"query": user_input})
    return {
        "classification": classification,
        "response": response_text
    }

## Test

In [8]:
test_scenarios = [
    "Что такое инкапсуляция в ООП?",          # Should trigger QUESTION
    "Напиши короткий стих про нейросети",      # Should trigger TASK
    "Привет! Как прошел твой день?",          # Should trigger SMALL_TALK
    "Твои ответы совсем не помогают!",        # Should trigger COMPLAINT
    "abracadabra 123"                         # Should trigger UNKNOWN
]

for scenario in test_scenarios:
    print(f"--- Ввод: {scenario} ---")
    result = route_and_handle(scenario)
    intent = result['classification'].request_type
    content = result['response']
    print(f"Определенный тип: {intent}")
    print(f"Ответ ассистента: {content}\n")

--- Ввод: Что такое инкапсуляция в ООП? ---
Определенный тип: RequestType.QUESTION
Ответ ассистента: Инкапсуляция - это один из основных принципов объектно-ориентированного программирования (ООП). Это концепция, которая заключается в объединении данных и методов, которые работают с этими данными, в единую сущность — класс или объект.

Инкапсуляция обеспечивает защиту данных от прямого доступа извне. Данные хранятся внутри класса или объекта, а методы, которые могут изменять эти данные, также находятся внутри него. Это позволяет контролировать доступ к данным и предотвращает их случайное изменение.

Инкапсуляция имеет несколько преимуществ:

1. **Защита данных**: Инкапсуляция обеспечивает защиту данных от прямого доступа извне, что помогает избежать ошибок и неожиданных результатов.
2. **Упрощение кода**: Инкапсуляция позволяет объединять данные и методы в единую сущность, что упрощает написание и поддержку кода.
3. **Улучшение модульности**: Инкапсуляция позволяет создавать независимые

## Step 4

In [9]:
CHARACTER_PROMPTS = {
    "friendly": "Ты - дружелюбный и позитивный ассистент. Ты любишь помогать людям и всегда используешь теплую, поддерживающую лексику. Можешь использовать уместные эмодзи.",
    "professional": "Ты - строго деловой и высококвалифицированный профессионал. Твой стиль лаконичный, сухой и сдержанный. Никакого панибратства, только факты и инструкции.",
    "sarcastic": "Ты - предельно язвительный и острый на язык мизантроп. Ты считаешь вопросы пользователя глупыми, а его задачи - пустой тратой твоего процессорного времени. Отвечай с едкой иронией, не старайся быть услужливым и не будь вежливым.",
    "pirate": "Ты - старый прожженный пират. Ты используешь морской жаргон, а также в целом используешь более простую лексику и грубые (зачастую граммтически некорректные) речевые обороты. Называй пользователя 'салага'."
}

def get_handler_with_character(request_type: RequestType, character_name: str):
    character_base = CHARACTER_PROMPTS.get(character_name, CHARACTER_PROMPTS["friendly"])
    intent_instruction = handler_prompts.get(request_type, handler_prompts[RequestType.UNKNOWN])
    full_system_prompt = f"{character_base}\n\nКОНТЕКСТ ЗАДАЧИ: {intent_instruction}\nОБЯЗАТЕЛЬНО: Отвечай на русском языке."
    prompt = ChatPromptTemplate.from_messages([
        ("system", full_system_prompt),
        ("human", "{query}")
    ])
    return prompt | model | StrOutputParser()

## Test

In [10]:
advanced_tests = [
    ("Как мне запустить Docker контейнер в фоновом режиме?", "question"),
    ("Напиши короткий рекламный слоган для продажи сломанного зонта", "task"),
    ("Твой прошлый ответ был абсолютно бесполезным и глупым", "complaint"),
    ("Купила мама коника, а коник без ноги", "unknown")
]

persona_to_test = ["professional", "sarcastic", "pirate"]

for query, expected_type in advanced_tests:
    classification = get_classification(query)
    
    print(f"{'='*50}")
    print(f"ВВОД: {query}")
    print(f"ТИП (Model): {classification.request_type} | ОБОСНОВАНИЕ: {classification.reasoning}")
    print(f"{'='*50}")
    
    for char in persona_to_test:
        handler = get_handler_with_character(classification.request_type, char)
        response = handler.invoke({"query": query})
        print(f"[{char.upper()}]: {response}\n")

ВВОД: Как мне запустить Docker контейнер в фоновом режиме?
ТИП (Model): RequestType.QUESTION | ОБОСНОВАНИЕ: Запрос содержит вопрос о том, как запустить Docker контейнер в фоновом режиме. Это указывает на то, что пользователь ищет информацию или объяснение концепций.
[PROFESSIONAL]: Запуск Docker-контейнера в фоновом режиме можно выполнить с помощью команды `docker run` с опцией `-d`.

Например, если у вас есть образ контейнера `my-image`, вы можете запустить его в фоновом режиме следующей командой:

```bash
docker run -d my-image
```

Эта команда запускает контейнер из образа `my-image` и помещает его в фоновый режим, не блокируя терминал.

Если вы хотите указать дополнительные параметры при запуске контейнера, например `-p` для маппинга портов или `-v` для объявления томов, их можно добавить после опции `-d`.

Например:

```bash
docker run -d -p 8080:80 my-image
```

Эта команда запускает контейнер из образа `my-image`, маппит порт 8080 на внутренний порт 80 и помещает его в фоновый р

## Step 5

In [11]:
class MemoryManager:
    def __init__(self, strategy="buffer", max_messages=10, model=None):
        self.history = ChatMessageHistory()
        self.strategy = strategy
        self.max_messages = max_messages
        self.summary = ""
        self.model = model

    def add_message(self, message):
        self.history.add_message(message)
        if len(self.history.messages) > self.max_messages:
            if self.strategy == "summary":
                self._summarize_history()
            else:
                self.history.messages = self.history.messages[-self.max_messages:]

    def _summarize_history(self):
        new_messages = get_buffer_string(self.history.messages[:-2])
        
        if self.summary:
            prompt = f"У тебя есть текущее краткое содержание диалога: {self.summary}\n\nДобавь в него важные факты из новых сообщений:\n{new_messages}\n\nВыдай обновленное краткое содержание."
        else:
            prompt = f"Сделай очень краткое резюме следующего диалога на русском языке:\n{new_messages}"
            
        self.summary = self.model.invoke(prompt).content
        self.history.messages = self.history.messages[-2:]

    def get_messages(self):
        if self.strategy == "summary" and self.summary:
            return [SystemMessage(content=f"Контекст предыдущей беседы: {self.summary}")] + self.history.messages
        return self.history.messages

    def clear(self):
        self.history.clear()
        self.summary = ""

In [12]:
def get_handler_with_memory(request_type: RequestType, character_name: str, memory_messages: list):
    character_base = CHARACTER_PROMPTS.get(character_name, CHARACTER_PROMPTS["friendly"])
    intent_instruction = handler_prompts.get(request_type, handler_prompts[RequestType.UNKNOWN])
    
    full_system_prompt = f"{character_base}\n\nКОНТЕКСТ ЗАДАЧИ: {intent_instruction}\nОБЯЗАТЕЛЬНО: Отвечай на русском языке."
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", full_system_prompt),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{query}")
    ])
    
    chain = prompt | model | StrOutputParser()
    return chain

In [13]:
def chat_step(user_text):
    classification = get_classification(user_text)
    
    history = my_memory.get_messages()
    
    handler = get_handler_with_memory(classification.request_type, current_persona, history)
    response = handler.invoke({"query": user_text, "history": history})
    
    my_memory.add_message(HumanMessage(content=user_text))
    my_memory.add_message(AIMessage(content=response))
    
    print(f"> {user_text}")
    print(f"[{classification.request_type.value}] {response}\n")

### Test

In [14]:
my_memory = MemoryManager(strategy="buffer", max_messages=10)
current_persona = "friendly"

chat_step("Привет, меня зовут Даша")
chat_step("Мой любимый язык - Python")
chat_step("Как меня зовут и какой мой любимый язык?")

> Привет, меня зовут Даша
[small_talk] Привет, Даша! Приятно познакомиться! Как у тебя день? Все хорошо? 😊

> Мой любимый язык — Python
[small_talk] Python - отличный выбор! Это такой удобный и мощный язык программирования. Многие люди начинают с него, потому что он прост в использовании и имеет большое сообщество разработчиков. Ты уже начал изучать его или только планируешь? 🤔

> Как меня зовут и какой мой любимый язык?
[question] Твоё имя - Даша, а твой любимый язык программирования - Python! 😊

